# Clase 10 — RAG: un chatbot que responde sobre tus propios documentos

## Objetivos de aprendizaje

- Entender qué es RAG (*Retrieval-Augmented Generation*) y qué problema resuelve.
- Reconocer las cinco piezas de un pipeline de RAG: loader, text splitter,
  embeddings, vector store y retriever + chain.
- Construir, con LangChain, Chroma y modelos gratuitos de Hugging Face, un
  chatbot que responde preguntas sobre un PDF propio.
- Ejecutar el pipeline completo en Colab sin pagar nada ni depender de OpenAI.

## Qué vamos a construir

Un "Personal Data Assistant": generamos un PDF de ejemplo con
`reportlab`, lo indexamos con embeddings y Chroma, y le hacemos preguntas
sobre su contenido usando un LLM gratuito. Es la misma arquitectura del
proyecto `wbphl-build_own_chatbot_without_open_ai` de IBM, en su variante
100% gratuita (`worker_huggingFace.py`).

## Apuntes: qué es RAG y cómo funciona

Un LLM genérico no sabe nada de un documento que no vio durante su
entrenamiento. Reentrenarlo cada vez que aparece un documento nuevo es
carísimo. **RAG** resuelve esto sin tocar el modelo: en vez de meterle el
documento entero, busca primero los fragmentos relevantes y se los entrega
como contexto justo antes de la pregunta — como darle a un becario solo las
páginas exactas del manual que necesita, en vez de todo el libro.

El pipeline tiene cinco piezas:

1. **Document loader**: carga el documento (aquí, `PyPDFLoader` para PDF).
2. **Text splitter**: trocea el texto en fragmentos manejables
   (`RecursiveCharacterTextSplitter`, con solape entre fragmentos).
3. **Embeddings**: convierte cada fragmento en un vector de significado
   (`sentence-transformers/all-MiniLM-L6-v2`).
4. **Vector store**: guarda esos vectores para buscar por similitud
   (**Chroma**).
5. **Retriever + chain**: busca los fragmentos más parecidos a la pregunta y
   se los pasa al LLM (`RetrievalQA`, de LangChain).

En el material original de IBM, el LLM es `WatsonxLLM` (de pago). Aquí
usamos la alternativa gratuita real del propio repositorio de IBM,
`HuggingFaceHub` con el modelo `tiiuae/falcon-7b-instruct` — solo necesita un
token gratuito de Hugging Face, sin tarjeta de crédito.

In [ ]:
# Instalamos las librerías necesarias: LangChain y sus integraciones,
# Chroma como base de datos vectorial, sentence-transformers para los
# embeddings, pypdf para leer PDFs, y reportlab para generar nuestro PDF de
# ejemplo.
!pip install -q langchain langchain-community chromadb sentence-transformers pypdf reportlab huggingface_hub

### Paso 0: creamos un PDF de ejemplo

En una app real, este PDF lo subiría el usuario (su propio manual, contrato,
informe, etc.). Aquí generamos uno nosotros mismos con `reportlab` para que
el notebook sea 100% autocontenido: una pequeña política de vacaciones
ficticia de una empresa, con datos concretos que luego le vamos a preguntar
al chatbot.

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

def crear_pdf_ejemplo(ruta="documento_ejemplo.pdf"):
    contenido = """Manual interno de Acme Corp - Politica de vacaciones y viajes

    Articulo 1. Dias de vacaciones.
    Todos los empleados a tiempo completo de Acme Corp tienen derecho a 22
    dias laborables de vacaciones pagadas por cada año completo trabajado.
    Los empleados a tiempo parcial reciben dias proporcionales a su jornada.

    Articulo 2. Solicitud de vacaciones.
    Las vacaciones deben solicitarse con al menos 15 dias de antelacion a
    traves del portal interno de Recursos Humanos. El responsable directo
    debe aprobar la solicitud en un plazo maximo de 5 dias habiles.

    Articulo 3. Reembolso de gastos de viaje.
    Los empleados que viajen por motivos de trabajo pueden solicitar el
    reembolso de transporte, alojamiento y comidas presentando los recibos
    originales dentro de los 30 dias siguientes al viaje. El limite diario
    para comidas es de 40 dolares.

    Articulo 4. Dias por enfermedad.
    Los empleados tienen derecho a 10 dias pagados por enfermedad al año,
    que no requieren justificante medico si son menos de 3 dias seguidos.
    """
    c = canvas.Canvas(ruta, pagesize=letter)
    texto = c.beginText(40, 750)
    texto.setFont("Helvetica", 11)
    for linea in contenido.strip().split("\n"):
        texto.textLine(linea.strip())
    c.drawText(texto)
    c.save()
    return ruta

ruta_pdf = crear_pdf_ejemplo()
print("PDF de ejemplo creado en:", ruta_pdf)

### Paso 1: el LLM y los embeddings gratuitos

`HuggingFaceHub` necesita un token gratuito de Hugging Face (te lo pide una
sola vez). Puedes crear uno en https://huggingface.co/settings/tokens con
una cuenta gratuita — no hace falta tarjeta de crédito.

In [ ]:
import os
from getpass import getpass

if "HUGGINGFACEHUB_API_TOKEN" not in os.environ:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass(
        "Pega tu token gratuito de Hugging Face (https://huggingface.co/settings/tokens): "
    )

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFaceHub

# Modelo de embeddings: convierte texto en vectores de significado.
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

# LLM gratuito de Hugging Face (alternativa real a WatsonxLLM en el
# material de IBM: worker_huggingFace.py usa exactamente este modelo).
llm = HuggingFaceHub(
    repo_id="tiiuae/falcon-7b-instruct",
    model_kwargs={"temperature": 0.5, "max_new_tokens": 200},
)

### Paso 2: cargar el PDF y trocearlo

`PyPDFLoader` extrae el texto del PDF. `RecursiveCharacterTextSplitter` lo
divide en fragmentos de 1024 caracteres, con 64 caracteres de solape entre
fragmentos consecutivos — los mismos valores que usa el `worker.py` real de
IBM.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = PyPDFLoader(ruta_pdf)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=64)
texts = text_splitter.split_documents(documents)

print(f"El PDF se dividio en {len(texts)} fragmentos.")
print("--- Primer fragmento ---")
print(texts[0].page_content[:300])

### Paso 3: indexar en Chroma y armar la cadena de RAG

Guardamos los fragmentos (convertidos en vectores) en Chroma, y construimos
un `retriever` que busca los 6 fragmentos más relevantes para cada pregunta
(`search_type="mmr"` evita que esos 6 fragmentos sean todos casi idénticos
entre sí). `RetrievalQA` con `chain_type="stuff"` mete esos fragmentos
directamente en el prompt del LLM.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA

db = Chroma.from_documents(texts, embeddings)

retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25})

chain = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=retriever, input_key="question"
)

### Paso 4: preguntando al chatbot sobre nuestro PDF

Igual que en `process_prompt` del `worker.py` real, guardamos el historial
de la conversación como una lista de pares (pregunta, respuesta).

In [ ]:
chat_history = []

def process_prompt(prompt):
    output = chain({"question": prompt, "chat_history": chat_history})
    answer = output["result"]
    chat_history.append((prompt, answer))
    return answer

respuesta = process_prompt("¿Cuantos dias de vacaciones tengo por año?")
print(respuesta)

In [ ]:
respuesta = process_prompt("¿Cual es el limite diario para comidas en un viaje de trabajo?")
print(respuesta)

## Referencia: la versión con watsonx (de pago) del material original

```python
# worker.py (fragmento real, requiere credenciales de watsonx)
from langchain_ibm import WatsonxLLM

def init_llm():
    params = {
        GenParams.MAX_NEW_TOKENS: 256,
        GenParams.TEMPERATURE: 0.1,
    }
    llm = WatsonxLLM(
        model_id="meta-llama/llama-4-maverick-17b-128e-instruct-fp8",
        url="https://us-south.ml.cloud.ibm.com",
        params=params,
        project_id=project_id,
    )
    embeddings = HuggingFaceInstructEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": DEVICE},
    )
    return llm, embeddings
```

La única diferencia real está en cómo se crea el LLM (`WatsonxLLM` frente a
`HuggingFaceHub`); el resto del pipeline —cargar, trocear, indexar,
recuperar, responder— es idéntico al que acabas de ejecutar.

## Resumen de la clase

- RAG deja que un LLM responda sobre documentos que nunca vio en su
  entrenamiento, sin reentrenarlo: recupera fragmentos relevantes y se los
  entrega como contexto.
- El pipeline: document loader → text splitter → embeddings → vector store
  → retriever + chain.
- `RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=64)` y
  `Chroma` son las piezas exactas que usa el material real de IBM.
- Con `HuggingFaceHub` (`falcon-7b-instruct`) y `HuggingFaceEmbeddings`
  (`all-MiniLM-L6-v2`) puedes montar todo el pipeline gratis, sin OpenAI ni
  watsonx.

Sigue con:
- Capítulo del manual: `manual/14_rag_documentos.md`
- Notebook de práctica: `notebooks_practica/clase10_practica_rag_documentos.ipynb`
  (ahí vas a reproducir los TODOs reales de `worker.py`/`Worker_completed.py`).